# Visão geral do pipeline climático

Este notebook percorre, passo a passo, a preparação de um ano de dados **fictícios** antes de qualquer modelagem. Cada etapa chama uma função do pacote `pipeline_climatico`, a mesma usada por `python -m pipeline_climatico tudo`, e mostra o que ela verifica ou altera.

Execute a partir da raiz do repositório ou da pasta `notebooks/`. Nenhum resultado aqui é científico: os dados foram inventados para exercitar o fluxo.

In [ ]:
import os
import sys
from pathlib import Path

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(raiz))
os.chdir(raiz)

import pandas as pd

from pipeline_climatico import ARQUIVO_ERA5, ARQUIVO_INMET, indice_horario_anual, inspecao, preparo

## 1. Inventário das variáveis

Antes de transformar qualquer coisa, liste o que chegou: nomes de colunas, tipos inferidos e quantos valores faltam em cada uma.

In [ ]:
pd.concat([inspecao.inventariar(ARQUIVO_INMET), inspecao.inventariar(ARQUIVO_ERA5)], ignore_index=True)

## 2. Validação da estrutura

`validar` interrompe o fluxo se faltar coluna, se houver data repetida ou fora de ordem. O passo mais comum entre registros revela a resolução de cada fonte: 1 h no INMET e 3 h no ERA5.

In [ ]:
pd.DataFrame([inspecao.validar(ARQUIVO_INMET), inspecao.validar(ARQUIVO_ERA5)])

## 3. Horários ausentes no INMET

O ano de 2024 tem 8 784 horas. O arquivo do INMET fictício tem menos linhas: algumas horas simplesmente não existem no bruto. Uma hora ausente é diferente de uma linha com valor vazio, e as duas precisam aparecer.

In [ ]:
inmet = pd.read_csv(ARQUIVO_INMET, parse_dates=["data_hora"])
grade = indice_horario_anual()
sem_registro = grade.difference(inmet["data_hora"])

print(f"horas esperadas: {len(grade)}")
print(f"linhas no bruto: {len(inmet)}")
print(f"horas sem registro: {len(sem_registro)}")
print(f"valores vazios: {int(inmet.isna().sum().sum())}")

## 4. Lacunas curtas e longas

A regra adotada interpola no tempo apenas lacunas internas de até 6 horas seguidas. Uma lacuna maior fica inteira como ausente, em vez de parcialmente preenchida. Veja a regra em uma série pequena com limite de 3 horas:

In [ ]:
indice = pd.date_range("2024-01-01", periods=10, freq="h")
serie = pd.Series([20, None, None, 23, 24, None, None, None, None, 29], index=indice, dtype=float)
tratada, imputado = preparo.preencher_lacunas_curtas(serie, max_horas=3)
pd.DataFrame({"bruto": serie, "tratado": tratada, "imputado": imputado})

Na base do INMET, temperatura e umidade seguem essa regra. A chuva **não** é interpolada nem trocada por zero, porque uma hora sem registro não é uma hora seca. Ela fica ausente e marcada em `precipitacao_inmet_mm_ausente`.

In [ ]:
inmet_tratado = preparo.preparar_inmet()
variaveis = ["temperatura_inmet_c", "umidade_relativa_inmet_pct", "precipitacao_inmet_mm"]
pd.DataFrame(
    {
        "vazias após inserir as horas": [
            int(inmet_tratado[f"{v}_imputado"].sum() + inmet_tratado[v].isna().sum())
            if f"{v}_imputado" in inmet_tratado
            else int(inmet_tratado[v].isna().sum())
            for v in variaveis
        ],
        "interpoladas": [int(inmet_tratado.get(f"{v}_imputado", pd.Series(dtype=int)).sum()) for v in variaveis],
        "mantidas ausentes": [int(inmet_tratado[v].isna().sum()) for v in variaveis],
    },
    index=variaveis,
)

## 5. Unidades e resolução do ERA5

O ERA5 fictício traz temperatura em kelvin e chuva em metros, acumulada em 3 horas. A conversão de unidade é simples. O alinhamento temporal pede mais cuidado: interpolar um **acumulado** como se fosse uma grandeza instantânea multiplica o total pelo número de passos.

In [ ]:
era5 = pd.read_csv(ARQUIVO_ERA5, parse_dates=["data_hora"]).set_index("data_hora")
chuva_3h_mm = era5["precipitacao_total_m"] * 1000

errado = chuva_3h_mm.reindex(grade).interpolate(method="time", limit_direction="both")
certo = preparo.preparar_era5()["precipitacao_era5_mm"]

pd.Series(
    {
        "total no bruto (mm)": chuva_3h_mm.sum(),
        "interpolando o acumulado (mm)": errado.sum(),
        "repartindo o acumulado em 3 h (mm)": certo.sum(),
    }
).round(1)

A temperatura, que é uma grandeza instantânea, pode ser interpolada para a grade horária. Os horários criados ficam marcados em `registro_era5_interpolado`.

Atenção: neste arquivo fictício, o acumulado de chuva cobre as 3 horas que **começam** no horário indicado. No ERA5 real, acumulados são atribuídos ao **fim** do período. Confira sempre a convenção da fonte antes de alinhar séries.

## 6. Base processada

A base final junta as duas fontes na mesma grade horária e mantém todas as marcações. Quem usar a base depois consegue separar dado observado, dado interpolado e dado ausente.

In [ ]:
base = preparo.preparar_base()
print(base.shape)
base.head()

In [ ]:
base.drop(columns="data_hora").describe().T

Para gravar a base em `data/processed/` e gerar os painéis em `figures/`, execute na raiz:

```bash
python -m pipeline_climatico preparar
python -m pipeline_climatico graficos
```